### Imports

**Run this cell every time you open the notebook.**

In [1]:
# Standard Library
import os
import sys
import csv
import shutil
import random
from pathlib import Path
from collections import Counter

# Scientific & Visualization 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Deep Learning
import torch
from ultralytics import YOLO

# Dataset & Detection
from roboflow import Roboflow
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

### Project Configuration & Directory Setup

**Run this cell every time you open the notebook.**

In [2]:
# Project Root (adjust to your local setup)
PROJECT_ROOT = os.path.expanduser("~/Documents/Thesis")

# Core Directories
DATA_DIR   = os.path.join(PROJECT_ROOT, "data")
RUNS_DIR   = os.path.join(PROJECT_ROOT, "runs")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

# Dataset Paths (XD38M DATASET)
DATASET_DIR       = os.path.join(DATA_DIR, "coral-detection-1")
FIXED_DATASET_DIR = os.path.join(DATA_DIR, "coral-detection-1-fixed")
YAML_PATH         = os.path.join(FIXED_DATASET_DIR, "data.yaml")

# Dataset Paths (CORAL-QUADRAT-2 DATASET)
CORAL_QUAD_DIR = os.path.join(DATA_DIR, "coral-quadrat-2")
FXD_CORAL_QUAD_DIR = os.path.join(DATA_DIR, "coral-quadrat-2-fixed")
FNL_CORAL_QUAD_DIR = os.path.join(DATA_DIR, "coral-quadrat-2-final")
CORAL_QUAD_YAML_PATH = os.path.join(FNL_CORAL_QUAD_DIR, "data.yaml")

# Training Run Name
RUN_NAME = "trained_model"  # Change this for each new run

# Derived Paths
BEST_WEIGHTS_PATH = os.path.join(RUNS_DIR, RUN_NAME, "weights", "best.pt")
VERIFY_DIR        = os.path.join(DATA_DIR, "verify_output")
VERIFY_CORAL_QUAD_DIR   = os.path.join(DATA_DIR, "verify_coral-quad_output")
# TEST_IMAGE_PATH   = os.path.join(PROJECT_ROOT, "test_images", "GOPR3275.jpg")  # ← change image as needed
CORALNET_TEST_DIR = Path(PROJECT_ROOT, "test_images")

# Create Directories
for d in [DATA_DIR, RUNS_DIR, MODELS_DIR, VERIFY_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Runs directory: {RUNS_DIR}")
print(f"Active run    : {RUN_NAME}")

Project root  : C:\Users\Administrator/Documents/Thesis
Data directory: C:\Users\Administrator/Documents/Thesis\data
Runs directory: C:\Users\Administrator/Documents/Thesis\runs
Active run    : trained_model


---
## Section 3: Dataset Download & Validation

### 3.1 Download XD38M Dataset *(run once)*

Downloads the XD38M coral detection dataset from Roboflow in YOLOv8 format. The dataset folder (`coral-detection-1`) is placed inside `DATA_DIR`.

In [3]:
rf = Roboflow(api_key="U2gyQmUUvtnxUfP40Mbs")
project = rf.workspace("alikhan-xtfbq").project("coral-detection-xd38m")
version = project.version(1)
dataset = version.download("yolov8", location=DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to C:\Users\Administrator/Documents/Thesis\data\coral-detection-1 in yolov8:: 100%|██████████| 5752/5752 [00:09<00:00, 620.02it/s] 


### 3.2 Validate Dataset Structure of XD38M

Confirm the expected train/valid/test folder layout and review the `data.yaml` configuration.

In [6]:
for split in ["train", "valid", "test"]:
    split_path = os.path.join(DATASET_DIR, split)
    print(f"\nContents of '{split}' folder:")
    print(os.listdir(split_path))

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
print(f"\nReading data.yaml from: {yaml_path}\n")
with open(yaml_path, "r") as f:
    print(f.read())


Contents of 'train' folder:
['images', 'labels']

Contents of 'valid' folder:
['images', 'labels']

Contents of 'test' folder:
['images', 'labels']

Reading data.yaml from: C:\Users\Administrator/Documents/Thesis\data\coral-detection-1\data.yaml

names:
- '-'
- This dataset was exported via roboflow.com on March 23- 2024 at 2-21 AM GMT
- coralreef - v6 2024-03-22 5-36am
nc: 3
roboflow:
  license: CC BY 4.0
  project: coral-detection-xd38m
  url: https://universe.roboflow.com/alikhan-xtfbq/coral-detection-xd38m/dataset/1
  version: 1
  workspace: alikhan-xtfbq
test: ../test/images
train: ../train/images
val: ../valid/images



### 3.3 Download coral-quadrat-2 Dataset for YOLO Fine-Tuning  *(run once)*

Downloads the coral-quadrat-2 Dataset from Roboflow in YOLOv8 format. The dataset folder (`coral-quadrat-2`) is placed inside `DATA_DIR`.

In [4]:
rf = Roboflow(api_key="U2gyQmUUvtnxUfP40Mbs")
project = rf.workspace("lifeformannotation").project("coral-quadrat-2")
version = project.version(9)
dataset = version.download("yolov8", location=CORAL_QUAD_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to C:\Users\Administrator/Documents/Thesis\data\coral-quadrat-2 in yolov8:: 100%|██████████| 1469/1469 [00:02<00:00, 547.52it/s]


### 3.2 Validate Dataset Structure of coral-quadrat-2

Confirm the expected train/valid/test folder layout and review the `data.yaml` configuration.

In [5]:
for split in ["train", "valid", "test"]:
    split_path = os.path.join(CORAL_QUAD_DIR, split)
    print(f"\nContents of '{split}' folder:")
    print(os.listdir(split_path))

yaml_path = os.path.join(CORAL_QUAD_DIR, "data.yaml")
print(f"\nReading data.yaml from: {yaml_path}\n")
with open(yaml_path, "r") as f:
    print(f.read())


Contents of 'train' folder:
['images', 'labels']

Contents of 'valid' folder:
['images', 'labels']

Contents of 'test' folder:
['images', 'labels']

Reading data.yaml from: C:\Users\Administrator/Documents/Thesis\data\coral-quadrat-2\data.yaml

names:
- Acropora Branching
- Acropora Tabulate
- Encrusting
- Foliose
- Massive
- Non-acropora branching
- Submassive
- mushroom
nc: 8
roboflow:
  license: CC BY 4.0
  project: coral-quadrat-2
  url: https://universe.roboflow.com/lifeformannotation/coral-quadrat-2/dataset/9
  version: 9
  workspace: lifeformannotation
test: ../test/images
train: ../train/images
val: ../valid/images



---
## Section 4: Dataset Label Analysis & Reconfiguration

The XD38M dataset from Roboflow ships with **3 class labels** (0, 1, 2), but this thesis requires a single class: **coral**. This section diagnoses the labeling issue and produces a clean, single-class copy of the dataset before training.

### 4.1 Count Class Distribution

Count how many annotations exist per class across all dataset splits.

In [7]:
# XD38M DATASET
def count_classes(labels_dir):
    counter = Counter()
    for txt_file in Path(labels_dir).glob("*.txt"):
        for line in txt_file.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    return counter

for split in ["train", "valid", "test"]:
    labels_path = Path(DATASET_DIR) / split / "labels"
    print(f"{split:6s}: {dict(count_classes(labels_path))}")

train : {1: 2749, 2: 3561, 0: 354}
valid : {1: 368, 2: 443, 0: 27}
test  : {0: 47, 2: 469, 1: 298}


In [6]:
# CORAL-QUADRAT-2 DATASET
def count_classes(labels_dir):
    counter = Counter()
    for txt_file in Path(labels_dir).glob("*.txt"):
        for line in txt_file.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    return counter

for split in ["train", "valid", "test"]:
    labels_path = Path(CORAL_QUAD_DIR) / split / "labels"
    print(f"{split:6s}: {dict(count_classes(labels_path))}")

train : {1: 381, 3: 509, 4: 980, 5: 1777, 7: 723, 0: 750, 6: 857, 2: 2287}
valid : {3: 27, 5: 77, 4: 41, 7: 37, 0: 47, 6: 27, 2: 116, 1: 13}
test  : {5: 65, 7: 34, 0: 34, 1: 15, 6: 23, 3: 19, 4: 34, 2: 115}


### 4.2 Visualize Class Boxes

Randomly sample `n` images from the training split and draw color-coded bounding boxes per class. Output images are saved to `VERIFY_DIR`.

| Color | Class |
|-------|-------|
| Red   | 0     |
| Green | 1     |
| Blue  | 2     |

In [9]:
# XD38M DATASET
def visualize_classes(images_dir, labels_dir, n=5,
                      colors={0: (0, 0, 255), 1: (0, 255, 0), 2: (255, 0, 0)}):
    label_files = list(Path(labels_dir).glob("*.txt"))
    random.shuffle(label_files)
    for lf in label_files[:n]:
        img_path = next(Path(images_dir).glob(lf.stem + ".*"), None)
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        for line in lf.read_text().splitlines():
            parts = line.split()
            cls = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
            x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            color = colors.get(cls, (255, 255, 255))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, str(cls), (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        out_path = os.path.join(VERIFY_DIR, f"verify_{lf.stem}.jpg")
        cv2.imwrite(out_path, img)
    print(f"Saved {n} sample images to: {VERIFY_DIR}")

visualize_classes(
    images_dir=os.path.join(DATASET_DIR, "train", "images"),
    labels_dir=os.path.join(DATASET_DIR, "train", "labels"),
)

Saved 5 sample images to: C:\Users\Administrator/Documents/Thesis\data\verify_output


In [12]:
# CORAL-QUADRAT-2 DATASET
def visualize_classes(images_dir, labels_dir, n=5,
                      colors={0: (0, 0, 255), 1: (0, 165, 255), 2: (0, 255, 255),
                              3: (0, 255, 0), 4: (255, 255, 0), 5: (255, 0, 0),
                              6: (255, 0, 255), 7: (128, 128, 128)}):
    label_files = list(Path(labels_dir).glob("*.txt"))
    random.shuffle(label_files)
    for lf in label_files[:n]:
        img_path = next(Path(images_dir).glob(lf.stem + ".*"), None)
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        for line in lf.read_text().splitlines():
            parts = line.split()
            cls = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
            x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            color = colors.get(cls, (255, 255, 255))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, str(cls), (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        out_path = os.path.join(VERIFY_CORAL_QUAD_DIR, f"verify_{lf.stem}.jpg")
        cv2.imwrite(out_path, img)
    print(f"Saved {n} sample images to: {VERIFY_CORAL_QUAD_DIR}")

visualize_classes(
    images_dir=os.path.join(CORAL_QUAD_DIR, "train", "images"),
    labels_dir=os.path.join(CORAL_QUAD_DIR, "train", "labels"),
)

Saved 5 sample images to: C:\Users\Administrator/Documents/Thesis\data\verify_addl_output


### 4.3 Verify Class 0 Boxes

Spot-check up to 15 images containing class-0 annotations across all splits. Output images are saved to `VERIFY_DIR/class0_verify/`.

In [13]:
# XD38M DATASET
class0_verify_dir = Path(VERIFY_DIR) / "class0_verify"
class0_verify_dir.mkdir(exist_ok=True)

checked = 0
for split in ["train", "valid", "test"]:
    images_dir = Path(DATASET_DIR) / split / "images"
    labels_dir = Path(DATASET_DIR) / split / "labels"

    for txt_file in labels_dir.glob("*.txt"):
        if checked >= 15:
            break
        lines = [l for l in txt_file.read_text().splitlines() if l.strip()]
        class0_lines = [l for l in lines if int(l.split()[0]) == 0]
        if not class0_lines:
            continue

        img_path = next(images_dir.glob(txt_file.stem + ".*"), None)
        if img_path is None:
            continue

        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        for line in class0_lines:
            parts = line.split()
            xc, yc, bw, bh = map(float, parts[1:5])
            x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
            x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(img, "cls0", (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        out_path = class0_verify_dir / f"cls0_{split}_{txt_file.stem}.jpg"
        cv2.imwrite(str(out_path), img)
        checked += 1

print(f"Saved {checked} class-0 verification images to: {class0_verify_dir}")

Saved 15 class-0 verification images to: C:\Users\Administrator\Documents\Thesis\data\verify_output\class0_verify


In [14]:
# CORAL-QUADRAT-2 DATASET
class_verify_dir = Path(VERIFY_CORAL_QUAD_DIR) / "class_spotcheck"
class_verify_dir.mkdir(exist_ok=True, parents=True)

SAMPLES_PER_CLASS = 3  # adjust as needed
colors = {0: (0, 0, 255), 1: (0, 165, 255), 2: (0, 255, 255),
          3: (0, 255, 0), 4: (255, 255, 0), 5: (255, 0, 0),
          6: (255, 0, 255), 7: (128, 128, 128)}

# Discover which classes exist instead of hardcoding a count
all_classes = sorted(count_classes(Path(CORAL_QUAD_DIR) / "train" / "labels").keys())

for target_cls in all_classes:
    checked = 0
    for split in ["train", "valid", "test"]:
        images_dir = Path(CORAL_QUAD_DIR) / split / "images"
        labels_dir = Path(CORAL_QUAD_DIR) / split / "labels"

        for txt_file in labels_dir.glob("*.txt"):
            if checked >= SAMPLES_PER_CLASS:
                break
            lines = [l for l in txt_file.read_text().splitlines() if l.strip()]
            target_lines = [l for l in lines if int(l.split()[0]) == target_cls]
            if not target_lines:
                continue

            img_path = next(images_dir.glob(txt_file.stem + ".*"), None)
            if img_path is None:
                continue

            img = cv2.imread(str(img_path))
            h, w = img.shape[:2]
            color = colors.get(target_cls, (0, 0, 255))
            for line in target_lines:
                parts = line.split()
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
                x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                cv2.putText(img, f"cls{target_cls}", (x1, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            out_path = class_verify_dir / f"cls{target_cls}_{split}_{txt_file.stem}.jpg"
            cv2.imwrite(str(out_path), img)
            checked += 1
        if checked >= SAMPLES_PER_CLASS:
            break
    print(f"Class {target_cls}: saved {checked} samples")

print(f"\nAll spot-check images saved to: {class_verify_dir}")

Class 0: saved 3 samples
Class 1: saved 3 samples
Class 2: saved 3 samples
Class 3: saved 3 samples
Class 4: saved 3 samples
Class 5: saved 3 samples
Class 6: saved 3 samples
Class 7: saved 3 samples

All spot-check images saved to: C:\Users\Administrator\Documents\Thesis\data\verify_addl_output\class_spotcheck


### Analysis Findings

- FOR XD38M DATASET
After reviewing the verification images above, it is confirmed that the XD38M dataset from Roboflow contains **corrupted class labels** across all 3 classes (0, 1, 2). All three map to coral-related structures. Since the thesis requires a single `coral` class, the next step merges all 3 classes into one unified label in a cloned copy of the dataset.

---

### 4.4 Clone & Remap Dataset Labels  *(run once)*

Creates `coral-detection-1-fixed/` by cloning the original and remapping all class IDs to `0` (coral). Choose the appropriate `class_map` based on your class-0 visual inspection above.

In [8]:
# XD38M DATASET
dataset_root = Path(DATASET_DIR)
fixed_root   = Path(FIXED_DATASET_DIR)

# Wipe and clone original (never edit original in place)
if fixed_root.exists():
    shutil.rmtree(fixed_root)
shutil.copytree(dataset_root, fixed_root)

# ── CHOOSE ONE ──────────────────────────────────────────────────────────────
# If class 0 visual check passed (≥ 8/15 on coral): include all three
class_map = {0: 0, 1: 0, 2: 0}

# If class 0 visual check failed (< 8/15 on coral): exclude class 0
# class_map = {1: 0, 2: 0}
# ────────────────────────────────────────────────────────────────────────────

remapped, dropped, skipped = 0, 0, 0

for split in ["train", "valid", "test"]:
    labels_dir = fixed_root / split / "labels"
    for txt_file in labels_dir.glob("*.txt"):
        new_lines = []
        for line in txt_file.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_cls = int(parts[0])
            if old_cls in class_map:
                new_lines.append(" ".join([str(class_map[old_cls])] + parts[1:]))
                remapped += 1
            else:
                dropped += 1  # class 0 boxes dropped if using second class_map
        txt_file.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))
        if not new_lines:
            skipped += 1

# Write clean single-class data.yaml
(fixed_root / "data.yaml").write_text(
    "train: ../train/images\n"
    "val:   ../valid/images\n"
    "test:  ../test/images\n\n"
    "nc: 1\n"
    "names: ['coral']\n"
)

print(f"Done.")
print(f"  Annotations remapped : {remapped}")
print(f"  Annotations dropped  : {dropped}")
print(f"  Empty label files    : {skipped}")
print(f"  Fixed dataset at     : {fixed_root}")

Done.
  Annotations remapped : 8316
  Annotations dropped  : 0
  Empty label files    : 0
  Fixed dataset at     : C:\Users\Administrator\Documents\Thesis\data\coral-detection-1-fixed


In [9]:
# CORAL-QUADRAT-2 DATASET
dataset_root = Path(CORAL_QUAD_DIR)
fixed_root   = Path(FXD_CORAL_QUAD_DIR)

# Wipe and clone original (never edit original in place)
if fixed_root.exists():
    shutil.rmtree(fixed_root)
shutil.copytree(dataset_root, fixed_root)

# ── CHOOSE ONE ──────────────────────────────────────────────────────────────
# All 8 classes are coral growth forms per data.yaml — include all by default
class_map = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}

# If a spot-check (Section 3.3) shows a specific class doesn't look like coral,
# exclude it here instead, e.g. to drop class 7 ("mushroom"):
# class_map = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}
# ────────────────────────────────────────────────────────────────────────────

remapped, dropped, skipped = 0, 0, 0

for split in ["train", "valid", "test"]:
    labels_dir = fixed_root / split / "labels"
    for txt_file in labels_dir.glob("*.txt"):
        new_lines = []
        for line in txt_file.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_cls = int(parts[0])
            if old_cls in class_map:
                new_lines.append(" ".join([str(class_map[old_cls])] + parts[1:]))
                remapped += 1
            else:
                dropped += 1  # boxes dropped if that class is excluded above
        txt_file.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))
        if not new_lines:
            skipped += 1

# Write clean single-class data.yaml
(fixed_root / "data.yaml").write_text(
    "train: ../train/images\n"
    "val:   ../valid/images\n"
    "test:  ../test/images\n\n"
    "nc: 1\n"
    "names: ['coral']\n"
)

print(f"Done.")
print(f"  Annotations remapped : {remapped}")
print(f"  Annotations dropped  : {dropped}")
print(f"  Empty label files    : {skipped}")
print(f"  Fixed dataset at     : {fixed_root}")

Done.
  Annotations remapped : 8988
  Annotations dropped  : 0
  Empty label files    : 26
  Fixed dataset at     : C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-fixed


### 4.5 Sanity Check (Fixed Dataset)

Verify that all annotations in `coral-detection-1-fixed` use only class `0` before training.

In [10]:
# XD38M DATASET
for split in ["train", "valid", "test"]:
    counter = Counter()
    for txt_file in (Path(FIXED_DATASET_DIR) / split / "labels").glob("*.txt"):
        for line in txt_file.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    print(f"{split:6s}: {dict(counter)}")

train : {0: 6664}
valid : {0: 838}
test  : {0: 814}


In [11]:
# CORAL-QUADRAT-2 DATASET
for split in ["train", "valid", "test"]:
    counter = Counter()
    for txt_file in (Path(FXD_CORAL_QUAD_DIR) / split / "labels").glob("*.txt"):
        for line in txt_file.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    print(f"{split:6s}: {dict(counter)}")

train : {0: 8264}
valid : {0: 385}
test  : {0: 339}


---
### 4.6 Cleanup Oversized "Habitat-Scale" Bounding Boxes in `coral-quadrat-2-fixed`

**Purpose**

Removes annotations where BOTH normalized width > 0.85 AND normalized height > 0.85.
These almost always represent an entire coral habitat/quadrat rather than a single
colony (Issue 1 identified during the coral-quadrat-2 dataset audit -- the highest
severity issue found).

This MUST be run before `step2_finetune.py`. Fine-tuning on the uncleaned dataset
risks teaching the detector that a huge coral-covered region is a single object,
which produces oversized boxes and unusable crops for the ResNet50 stage later.

**Safety**

This script does NOT modify your dataset in place. It copies `coral-quadrat-2-fixed`
into a new folder (`coral-quadrat-2-cleaned`) and only removes boxes in the COPY, so
the original dataset is preserved if you ever need to re-inspect what was removed.

**Also included** (informational only, not auto-fixed)

A nested-bounding-box counter (Issue 4 from the audit). Nested boxes can reflect
legitimate overlapping colonies, so this script only reports a count for your
documentation log rather than deleting anything automatically.

**Requirements**

No extra installs needed beyond Python's standard library.

**Outputs** (all written inside the new cleaned dataset folder)

- `coral-quadrat-2-cleaned/` — the new, safe-to-train-on dataset
- `cleanup_manifest.csv` — every removed box logged individually
- `cleanup_summary.txt` — per-split counts, for your thesis appendix


In [12]:
SRC_ROOT = Path(FXD_CORAL_QUAD_DIR)
DST_ROOT = Path(FNL_CORAL_QUAD_DIR) 

# A box with BOTH normalized width and height above this is treated as a
# habitat-scale annotation and dropped, per the audit recommendation.
WH_THRESHOLD = 0.85

# A box pair where one box is at least this fraction contained inside another
# is flagged as "nested" for informational reporting only.
NESTED_CONTAINMENT_THRESHOLD = 0.90

SPLITS = ["train", "valid", "test"]
# ===================================================================================


def copy_dataset(src: Path, dst: Path) -> None:
    if dst.exists():
        print(f"[skip] {dst} already exists -- using existing copy. "
              f"Delete it manually first if you want a completely fresh copy.")
        return
    print(f"Copying dataset:\n  {src}\n  -> {dst}")
    shutil.copytree(src, dst)
    print("Copy complete.\n")


def fix_data_yaml_paths(dst_root: Path, old_root: Path) -> None:
    """data.yaml often stores an absolute 'path:' pointing at the ORIGINAL folder.
    If we don't fix this, YOLO will silently keep reading from coral-quadrat-2-fixed
    even though you think you're pointing it at the cleaned copy."""
    yaml_path = dst_root / "data.yaml"
    if not yaml_path.exists():
        print(f"[warn] no data.yaml found at {yaml_path} -- you may need to create/edit one manually.")
        return
    text = yaml_path.read_text()
    if str(old_root) in text:
        text = text.replace(str(old_root), str(dst_root))
        yaml_path.write_text(text)
        print(f"Updated data.yaml paths: {old_root} -> {dst_root}")
    else:
        print("data.yaml did not contain the old root path -- no path substitution needed "
              "(it may already use relative paths, which is fine).")


def yolo_to_xyxy(cx, cy, bw, bh):
    x1 = cx - bw / 2
    y1 = cy - bh / 2
    x2 = cx + bw / 2
    y2 = cy + bh / 2
    return x1, y1, x2, y2


def containment_ratio(inner, outer):
    ix1, iy1, ix2, iy2 = inner
    ox1, oy1, ox2, oy2 = outer
    ix1i, iy1i = max(ix1, ox1), max(iy1, oy1)
    ix2i, iy2i = min(ix2, ox2), min(iy2, oy2)
    if ix2i <= ix1i or iy2i <= iy1i:
        return 0.0
    inter_area = (ix2i - ix1i) * (iy2i - iy1i)
    inner_area = (ix2 - ix1) * (iy2 - iy1)
    if inner_area <= 0:
        return 0.0
    return inter_area / inner_area


def count_nested_boxes(dst_root: Path, splits: list, threshold: float) -> int:
    """Informational only -- does not modify any files."""
    nested_count = 0
    for split in splits:
        labels_dir = dst_root / split / "labels"
        if not labels_dir.exists():
            continue
        for txt_file in labels_dir.glob("*.txt"):
            boxes = []
            for line in txt_file.read_text().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) < 5:
                    continue
                cx, cy, bw, bh = map(float, parts[1:5])
                boxes.append(yolo_to_xyxy(cx, cy, bw, bh))
            for i in range(len(boxes)):
                for j in range(len(boxes)):
                    if i == j:
                        continue
                    if containment_ratio(boxes[i], boxes[j]) >= threshold:
                        nested_count += 1
                        break  # count each inner box once, even if nested in multiple boxes
    return nested_count


def clean_split(dst_root: Path, split: str, threshold: float, manifest_rows: list) -> dict:
    labels_dir = dst_root / split / "labels"
    if not labels_dir.exists():
        print(f"[warn] {labels_dir} not found -- skipping split '{split}'")
        return {"images_seen": 0, "boxes_seen": 0, "boxes_removed": 0}

    images_seen = 0
    boxes_seen = 0
    boxes_removed = 0

    for txt_file in sorted(labels_dir.glob("*.txt")):
        images_seen += 1
        lines = txt_file.read_text().splitlines()
        filtered = []

        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) < 5:
                print(f"[warn] malformed line in {txt_file.name}: '{line}' -- kept as-is")
                filtered.append(line)
                continue

            cls = parts[0]
            bw, bh = float(parts[3]), float(parts[4])
            boxes_seen += 1

            if bw > threshold and bh > threshold:
                boxes_removed += 1
                manifest_rows.append({
                    "split": split,
                    "image": txt_file.stem,
                    "class": cls,
                    "width": bw,
                    "height": bh,
                })
                continue  # drop this oversized habitat box

            filtered.append(line)

        txt_file.write_text("\n".join(filtered) + ("\n" if filtered else ""))

    return {"images_seen": images_seen, "boxes_seen": boxes_seen, "boxes_removed": boxes_removed}


def main():
    copy_dataset(SRC_ROOT, DST_ROOT)
    fix_data_yaml_paths(DST_ROOT, SRC_ROOT)

    manifest_rows = []
    summary_lines = ["Oversized Bounding Box Cleanup Summary", "=" * 40, ""]
    total_removed = 0
    total_boxes = 0

    for split in SPLITS:
        stats = clean_split(DST_ROOT, split, WH_THRESHOLD, manifest_rows)
        total_removed += stats["boxes_removed"]
        total_boxes += stats["boxes_seen"]
        line = (f"{split:6s} | images: {stats['images_seen']:4d} | "
                f"boxes: {stats['boxes_seen']:5d} | removed: {stats['boxes_removed']:3d}")
        print(line)
        summary_lines.append(line)

    pct = (total_removed / total_boxes * 100) if total_boxes else 0.0
    summary_lines.append("")
    summary_lines.append(f"TOTAL boxes removed: {total_removed} / {total_boxes} ({pct:.2f}%)")

    print(f"\nRunning informational nested-box scan (not modifying anything)...")
    nested_count = count_nested_boxes(DST_ROOT, SPLITS, NESTED_CONTAINMENT_THRESHOLD)
    summary_lines.append(f"Nested boxes flagged (informational, >= {NESTED_CONTAINMENT_THRESHOLD:.0%} "
                          f"containment, NOT removed): {nested_count}")
    print(f"Nested boxes flagged (informational only): {nested_count}")

    manifest_path = DST_ROOT / "cleanup_manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "image", "class", "width", "height"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    summary_path = DST_ROOT / "cleanup_summary.txt"
    summary_path.write_text("\n".join(summary_lines))

    print("\n" + "\n".join(summary_lines[-2:]))
    print(f"\nManifest saved to: {manifest_path}")
    print(f"Summary saved to:  {summary_path}")
    print(f"\nCleaned dataset ready at: {DST_ROOT}")
    print("Point step1_preflight_check.py and step2_finetune.py at THIS folder, not the original.")


main()


Copying dataset:
  C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-fixed
  -> C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-final
Copy complete.

data.yaml did not contain the old root path -- no path substitution needed (it may already use relative paths, which is fine).
train  | images:  674 | boxes:  8264 | removed:  49
valid  | images:   29 | boxes:   385 | removed:   2
test   | images:   29 | boxes:   339 | removed:   1

Running informational nested-box scan (not modifying anything)...
Nested boxes flagged (informational only): 301

TOTAL boxes removed: 52 / 8988 (0.58%)
Nested boxes flagged (informational, >= 90% containment, NOT removed): 301

Manifest saved to: C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-final\cleanup_manifest.csv
Summary saved to:  C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-final\cleanup_summary.txt

Cleaned dataset ready at: C:\Users\Administrator\Documents\Thesis\data\coral-quadrat-2-final
Point 